# Lezione 1 — Com'è fatta l'atmosfera
**Fisica dell'Atmosfera, Clima e Salute** · Dr. Piero Chiacchiaretta

**Come si usa questo notebook**
1. Ogni riquadro grigio è una *cella di codice*: cliccaci dentro e premi **Maiusc + Invio** (oppure il pulsante ▶).
2. Esegui le celle **in ordine**, dall'alto verso il basso.
3. Dove vedi `TODO` devi scrivere tu il codice, poi riesegui quella cella e la cella di controllo.

Non devi installare nulla: tutto gira nel browser.

In [ ]:
# --- Cella 1: preparazione (eseguila sempre per prima) ---
import math
import importlib, subprocess, sys

def _richiedi(pacchetto):
    try:
        return importlib.import_module(pacchetto)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacchetto])
        return importlib.import_module(pacchetto)

np = _richiedi("numpy")
_richiedi("matplotlib")
import matplotlib.pyplot as plt

# Costanti fisiche
G = 9.80665       # accelerazione di gravità [m/s^2]
R_AIR = 287.05    # costante specifica dell'aria secca [J/(kg K)]
P0 = 1013.25      # pressione al livello del mare [hPa]
print("Tutto pronto!")

## Esercizi 1–3 e sfida
Completa le funzioni sostituendo `raise NotImplementedError` con il tuo codice.

* **Es. 1** – altezza di scala $H = R\,T/g$ (in metri)
* **Es. 2** – pressione in un'atmosfera isoterma $P(z) = P_0\,e^{-z/H}$
* **Es. 3** – punto di rugiada dalla temperatura e dall'umidità relativa
* **Sfida** – base delle nubi: circa $125\,(T - T_d)$ metri

In [ ]:
def scale_height(T):
    """Altezza di scala H = R T / g in METRI (T in kelvin)."""
    # TODO: restituisci R_AIR * T / G
    raise NotImplementedError


def p_isothermal(z, T=288.15):
    """Pressione in hPa a quota z (in METRI) in un'atmosfera isoterma."""
    # TODO: P = P0 * exp(-z / H), con H = scale_height(T)   (usa math.exp)
    raise NotImplementedError


def e_sat(T_c):
    """Pressione di vapore saturo in hPa (formula di Magnus), T in °C. (Già fatta.)"""
    return 6.112 * math.exp(17.62 * T_c / (243.12 + T_c))


def dew_point(T_c, RH):
    """Punto di rugiada in °C. RH è una frazione (0.6 = 60%)."""
    # TODO: 1) e = RH * e_sat(T_c)
    #       2) g = math.log(e / 6.112)
    #       3) restituisci 243.12 * g / (17.62 - g)
    raise NotImplementedError


def cloud_base(T_c, RH):
    """Quota della base delle nubi in metri: ~125 m per ogni kelvin di (T - Td)."""
    # TODO: restituisci 125 * (T_c - dew_point(T_c, RH))
    raise NotImplementedError

### Controllo automatico
Esegui questa cella dopo aver completato le funzioni: `[OK]` = giusto, `[ ]` = da completare, `[X]` = da rivedere.

In [ ]:
def _controlla(nome, funzione, atteso, tol):
    try:
        val = funzione()
    except NotImplementedError:
        print(f"  [ ]  {nome}: da completare"); return False
    except Exception as err:
        print(f"  [X]  {nome}: errore nel codice -> {type(err).__name__}: {err}"); return False
    ok = abs(val - atteso) <= tol
    print(f"  {'[OK]' if ok else '[X] '} {nome}: ottenuto {val:.2f}   (atteso circa {atteso:.2f})")
    return ok

risultati = [
    _controlla("Es.1  H a 288.15 K [m]", lambda: scale_height(288.15), 8434, 10),
    _controlla("Es.2  P a 5000 m [hPa]", lambda: p_isothermal(5000), 560, 3),
    _controlla("Es.2  P a 8848 m [hPa]", lambda: p_isothermal(8848), 355, 3),
    _controlla("Es.3  Td (T=25, U=60%) [°C]", lambda: dew_point(25, 0.6), 16.7, 0.2),
    _controlla("Es.3  Td (T=30, U=80%) [°C]", lambda: dew_point(30, 0.8), 26.2, 0.2),
    _controlla("Sfida base nubi (T=30, U=50%) [m]", lambda: cloud_base(30, 0.5), 1445, 20),
]
print(f"\nEsercizi corretti: {sum(risultati)} su {len(risultati)}")

## Simulazione 1 — Pressione e altezza di scala
Cambia i valori di `temperature` (in kelvin) e riesegui: come cambia la curva? Cosa succede a $H$?
*(Questa cella funziona solo dopo aver completato gli Esercizi 1 e 2.)*

In [ ]:
temperature = [250, 288.15, 320]          # <-- prova a modificarle

z = np.linspace(0, 30000, 300)            # quota in metri
plt.figure(figsize=(6, 5))
for T in temperature:
    p = [p_isothermal(x, T) for x in z]
    plt.plot(p, z / 1000, label=f"T = {T:.0f} K,  H = {scale_height(T)/1000:.1f} km")
plt.axhline(8.848, color="grey", ls=":"); plt.text(600, 9.2, "Everest")
plt.xlabel("Pressione [hPa]"); plt.ylabel("Quota [km]")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Simulazione 2 — Punto di rugiada e nubi
Per una data temperatura, come cambia il punto di rugiada al variare dell'umidità relativa?
*(Richiede l'Esercizio 3.)*

In [ ]:
T = 30                                     # <-- prova a cambiare la temperatura (°C)
umidita = np.arange(10, 101, 5)            # da 10% a 100%

td = [dew_point(T, u / 100) for u in umidita]
plt.figure(figsize=(6, 4))
plt.plot(umidita, td, "o-")
plt.axhline(T, color="grey", ls="--", label=f"T = {T} °C")
plt.xlabel("Umidità relativa [%]"); plt.ylabel("Punto di rugiada [°C]")
plt.legend(); plt.grid(alpha=0.3); plt.show()
print("A U = 100% il punto di rugiada coincide con la temperatura: l'aria è satura.")

## Compiti per la Lezione 2
1. Completa tutti gli esercizi (controllo: **6 su 6**) e salva il notebook: *File → Scarica → Scarica .ipynb*.
2. Rifletti: *se l'atmosfera non assorbisse l'infrarosso, quale sarebbe la temperatura della Terra?*
3. Porta 3 domande sull'effetto serra.